# Implied Volatility Surface Prediction — Final Submission
**Finance Club, IIT Roorkee**

## Approach in one paragraph
An implied-volatility surface is extremely smooth *along each option type*. The key insight driving
this solution is that **calls (CE) and puts (PE) lie on two separate smooth curves with a jump
between them at the money** — they are not one continuous smile. So the core predictor fits the
smile **separately for CE and PE** at each timestamp, by linear interpolation across log-moneyness,
using only other strikes observed at the *same* timestamp (allowed by the rules, no lookahead).
A small 5% blend with same-strike time interpolation handles the rare cases with sparse strikes.
This is a pure, financially-grounded interpolation method: fast, fully reproducible, and robust to
the public/private split because it has almost nothing to overfit.

**Validation MSE (random 20% cell hold-out, 6 seeds): ~9e-5** — a 1.9x improvement over treating CE+PE as one curve (~1.7e-4).

### Why CE/PE separation is the whole game
At any timestamp the put curve and the call curve sit at different IV levels and meet with a kink
near the spot price (a consequence of how puts and calls are quoted, not put-call parity). If you
interpolate across all 28 strikes as a single curve, every prediction near that boundary is dragged
across the jump — and near expiry, where IV is large, those boundary errors reach 0.10+ vol points
and dominate the squared error. Fitting CE and PE independently removes this entirely.

## 1. Imports and configuration

In [3]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_PATH = '/kaggle/input/finclub-open-project-26/dataset.csv'   
W_TIME    = 0.05            # weight on time-interpolation; 0.95 goes to the smile fit

## 2. Load data and identify strikes / option type

In [5]:
df = pd.read_csv(DATA_PATH)
df['datetime'] = pd.to_datetime(df['datetime'], format='%d-%m-%Y %H:%M')
df = df.sort_values('datetime').reset_index(drop=True)

opt_cols = [c for c in df.columns if c.endswith('CE') or c.endswith('PE')]

def get_strike(col):
    return int(col.replace('NIFTY27JAN26', '').replace('CE', '').replace('PE', ''))

strike_map = {c: get_strike(c) for c in opt_cols}
type_map   = {c: (1 if c.endswith('CE') else 0) for c in opt_cols}   # 1 = call, 0 = put

print(f'{df.shape[0]} timestamps, {len(opt_cols)} option columns')
print(f'Missing cells to predict: {int(df[opt_cols].isna().sum().sum())}')

975 timestamps, 28 option columns
Missing cells to predict: 5460


## 3. Reshape to long format
Each (timestamp, strike) becomes one row. `log_mny = log(strike / spot)` is the natural x-axis of
the volatility smile.

In [7]:
records = []
for i, r in df.iterrows():
    u = r['underlying_price']
    t = r['datetime']
    for c in opt_cols:
        records.append((i, t, c, strike_map[c], type_map[c], u, r[c]))

L = pd.DataFrame(records, columns=['ti','datetime','col','strike','is_call','under','iv'])
L['log_mny'] = np.log(L['strike'] / L['under'])

observed = L.dropna(subset=['iv']).copy()
missing  = L[L['iv'].isna()].copy()
print(f'Observed: {len(observed)}   Missing: {len(missing)}')

Observed: 21840   Missing: 5460


## 4. The two predictors

### 4a. Smile interpolation — fit CE and PE separately (the core idea)
At each timestamp we group by option *type* as well, so calls and puts never mix. Within each group
we interpolate IV linearly across log-moneyness. Linear (not cubic) keeps it stable in the volatile
near-expiry region, where higher-order fits overshoot.

In [9]:
def strike_interp_separate(train_df, target_df):
    preds = np.full(len(target_df), np.nan)
    fallback = train_df['iv'].median()
    # one interpolator per (timestamp, option-type)
    curves = {}
    for key, s in train_df.groupby(['ti', 'is_call']):
        s = s.sort_values('log_mny')
        ux, idx = np.unique(s['log_mny'].values, return_index=True)
        curves[key] = (ux, s['iv'].values[idx])
    tg = target_df.reset_index()
    for key, sub in tg.groupby(['ti', 'is_call']):
        ridx = sub.index.values
        c = curves.get(key)
        if c is None or len(c[0]) < 2:
            preds[ridx] = fallback
            continue
        f = interp1d(c[0], c[1], kind='linear', fill_value='extrapolate', bounds_error=False)
        preds[ridx] = f(sub['log_mny'].values)
    return preds

### 4b. Time interpolation — same strike, across timestamps
A light safety net for timestamps where a type has too few observed strikes to interpolate well.

In [11]:
def time_interp(train_df, target_df):
    preds = np.full(len(target_df), np.nan)
    fallback = train_df['iv'].median()
    series = {c: (s.sort_values('ti')['ti'].values.astype(float),
                  s.sort_values('ti')['iv'].values)
              for c, s in train_df.groupby('col')}
    tg = target_df.reset_index()
    for c, sub in tg.groupby('col'):
        ridx = sub.index.values
        g = series.get(c)
        if g is None or len(g[0]) < 2:
            preds[ridx] = fallback
            continue
        f = interp1d(g[0], g[1], kind='linear', fill_value='extrapolate', bounds_error=False)
        preds[ridx] = f(sub['ti'].values.astype(float))
    return preds

def predict(train_df, target_df, w_time=W_TIME):
    s = strike_interp_separate(train_df, target_df)
    t = time_interp(train_df, target_df)
    return np.clip((1 - w_time) * s + w_time * t, 1e-3, None)   # IV is strictly positive

## 5. Validation
We hide a random fraction of *observed* cells, predict them from the rest, and measure MSE.
Repeating over several seeds shows the score is stable. (The real task predicts from ~80% observed
data, which is denser than these splits, so the true score is typically a touch better.)
We also report the score of the naive single-curve interpolation to show what the CE/PE split buys.

In [13]:
def mse(a, b):
    return np.mean((np.asarray(a) - np.asarray(b)) ** 2)

def strike_interp_combined(train_df, target_df):   # naive baseline: CE+PE as ONE curve
    preds = np.full(len(target_df), np.nan); fb = train_df['iv'].median(); curves = {}
    for ti, s in train_df.groupby('ti'):
        s = s.sort_values('log_mny'); ux, ix = np.unique(s['log_mny'].values, return_index=True)
        curves[ti] = (ux, s['iv'].values[ix])
    tg = target_df.reset_index(drop=True)
    for ti, sub in tg.groupby('ti'):
        ri = sub.index.values; c = curves.get(ti)
        if c is None or len(c[0]) < 2: preds[ri] = fb; continue
        f = interp1d(c[0], c[1], kind='linear', fill_value='extrapolate', bounds_error=False)
        preds[ri] = f(sub['log_mny'].values)
    return np.clip(preds, 1e-3, None)

# Validation: hold out a random 20% of *cells* (matches the real 20% missingness),
# averaged over several seeds. The near-expiry (27th) cells are few but carry almost
# all the error, so single splits are noisy -> we average to get an honest number.
obs = observed.reset_index(drop=True).copy()
obs['cellid'] = obs['ti'].astype(str) + '_' + obs['col']
sep_s, comb_s, normal_s, spike_s = [], [], [], []
for seed in range(6):
    val_ids = set(obs['cellid'].sample(frac=0.20, random_state=seed))
    mask = obs['cellid'].isin(val_ids)
    tr, va = obs[~mask], obs[mask]
    psep = predict(tr, va); pcomb = strike_interp_combined(tr, va)
    sep_s.append(mse(va['iv'], psep)); comb_s.append(mse(va['iv'], pcomb))
    sp = va['datetime'].dt.day.values == 27
    normal_s.append(mse(va['iv'][~sp], psep[~sp])); spike_s.append(mse(va['iv'][sp], psep[sp]))

print(f'Separate CE/PE (this model): MSE = {np.mean(sep_s):.2e}   (std {np.std(sep_s):.1e})')
print(f'Combined naive baseline:     MSE = {np.mean(comb_s):.2e}')
print(f'Improvement factor:          {np.mean(comb_s)/np.mean(sep_s):.1f}x')
print(f'  -> normal-day cells:  MSE = {np.mean(normal_s):.2e}  (near-perfect)')
print(f'  -> expiry-day cells:  MSE = {np.mean(spike_s):.2e}  (dominates total error)')

Separate CE/PE (this model): MSE = 9.13e-05   (std 3.6e-05)
Combined naive baseline:     MSE = 1.69e-04
Improvement factor:          1.9x
  -> normal-day cells:  MSE = 8.66e-06  (near-perfect)
  -> expiry-day cells:  MSE = 1.13e-03  (dominates total error)


## 6. Fit on all observed data and predict the missing cells

In [15]:
final_pred = predict(observed, missing)
missing = missing.copy()
missing['pred'] = final_pred
print('Predicted', len(missing), 'cells')
print('min %.4f | median %.4f | max %.4f' % (final_pred.min(), np.median(final_pred), final_pred.max()))

# sanity: near-expiry (27th) predictions should be elevated
m27 = missing['datetime'].dt.day == 27
print('median pred  near-expiry (27th): %.3f   |   other days: %.3f'
      % (missing[m27]['pred'].median(), missing[~m27]['pred'].median()))

Predicted 5460 cells
min 0.0245 | median 0.1314 | max 5.7115
median pred  near-expiry (27th): 0.665   |   other days: 0.128


## 7. Build the submission (matches the organizers' grader)
The grader reads a **`filled_dataset.csv`** — the original wide file with every missing cell filled —
and turns each originally-missing cell into a row with `id = "{datetime}||{column}"` and column
`value`. So we (a) write `filled_dataset.csv` in the original row order, then (b) reproduce the
grader's `generate_solution` logic to write the final `submission.csv`.

In [17]:
# (a) write filled_dataset.csv : original wide file, missing cells filled, ORIGINAL row order
final_pred = predict(observed, missing)
missing = missing.copy(); missing['pred'] = np.round(final_pred, 6)  # 6 dp: Excel-safe, MSE impact ~1e-13

orig = pd.read_csv(DATA_PATH)                 # fresh copy => exact original order & datetime strings
opt_cols = [c for c in orig.columns if c != 'datetime']
filled = orig.copy()
# our df was sorted by datetime; the source is already sorted, so row index == original index.
for _, row in missing.iterrows():
    filled.iat[int(row['ti']), filled.columns.get_loc(row['col'])] = row['pred']
assert filled[opt_cols].isna().sum().sum() == 0
assert (filled['datetime'] == orig['datetime']).all()
filled.to_csv('filled_dataset.csv', index=False)

# (b) organizers' grader logic -> submission.csv  (id = "{datetime}||{col}", column = value)
SEPARATOR = '||'
rows = []
for col in opt_cols:
    for idx in orig.index[orig[col].isna()]:
        rows.append({'id': f"{orig.loc[idx,'datetime']}{SEPARATOR}{col}",
                     'value': filled.loc[idx, col]})
submission = pd.DataFrame(rows, columns=['id','value']).sort_values('id').reset_index(drop=True)
submission.to_csv('submission.csv', index=False)
print('Wrote filled_dataset.csv and submission.csv (%d rows)' % len(submission))
print(submission.head())

Wrote filled_dataset.csv and submission.csv (5460 rows)
                                      id     value
0  07-01-2026 09:15||NIFTY27JAN2624100PE  0.163507
1  07-01-2026 09:15||NIFTY27JAN2625500CE  0.114849
2  07-01-2026 09:15||NIFTY27JAN2625800CE  0.099896
3  07-01-2026 09:20||NIFTY27JAN2624000PE  0.170063
4  07-01-2026 09:20||NIFTY27JAN2624200PE  0.159692
